# Experiment 09 Analysis: Bioactivity Prediction vs Embedding Size

This notebook analyzes the results of Experiment 09, which systematically compares two molecular representation methods (Morgan fingerprints, HDC) across varying embedding sizes for similarity-based bioactivity prediction.

**Key Research Questions:**
1. How does embedding size affect virtual screening performance (AUC, BEDROC)?
2. Which representation method achieves better early recognition at different dimensionalities?
3. At what embedding size do we see diminishing returns for bioactivity prediction?
4. How do enrichment factors (EF1%, EF5%) scale with embedding dimensionality?

**Primary Metrics:**
- AUC: Area under ROC curve (overall ranking quality) - higher is better
- BEDROC(α=20): Early recognition metric for virtual screening - higher is better
- EF1%/EF5%: Enrichment factors at 1% and 5% - higher is better

In [ ]:
%matplotlib inline
import os
import time
import json
import datetime
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Latex
from rich.pretty import pprint
from pycomex.utils import is_experiment_archive
from pycomex.functional.experiment import Experiment

# This will be the path to the directory in which the notebook is located.
PATH: str = os.getcwd()
# This will have to be the path to the pycomex "results" directory containing the 
# experiment archives of interest.
RESULTS_PATH: str = os.path.join(PATH, 'results')

## Experiment Selection and Sorting Functions

Define how to filter and group experiments based on their parameters.

In [ ]:
# Filter which experiments will be loaded based on their name and/or parameters.
def select_experiment(experiment_name: str,
                      experiment_metadata: dict,
                      experiment_parameters: dict
                      ) -> bool:
    """Select only Experiment 09 bioactivity experiments."""
    main_condition = 'ex_09_a' in experiment_parameters.get('__PREFIX__', '')
    return main_condition


# Assign a unique key to the experiment based on its data / parameters etc.
def sort_experiment(experiment: Experiment) -> tuple:
    """Sort experiments by (encoding, embedding_size)."""
    
    # Determine encoding type from experiment name
    experiment_name = experiment.metadata['name']
    print(f'  Processing: {experiment_name}')
    
    if '_fp' in experiment_name or '__fp' in experiment_name:
        encoding = 'fp'
    elif '_hdc' in experiment_name or '__hdc' in experiment_name:
        encoding = 'hdc'
    else:
        encoding = 'unknown'
    
    # Extract embedding size based on encoding type
    if encoding == 'fp':
        embedding_size = experiment.parameters.get('FP_SIZE', 0)
    elif encoding == 'hdc':
        embedding_size = experiment.parameters.get('EMBEDDING_SIZE', 0)
    else:
        embedding_size = 0
    
    return (encoding, embedding_size)

## Experiment Discovery

Find all archived experiments in the results directory.

In [ ]:
# This list will contain the paths to the individual experiment *namespaces* which in 
# turn contain the actual individual experiment archives.
experiment_namespace_paths: list[str] = [
    path
    for file_name in os.listdir(RESULTS_PATH)
    if os.path.isdir(path := os.path.join(RESULTS_PATH, file_name))
]

# Subsequently, this list will contain the paths to the individual experiment archives
# folders.
experiment_paths: list[str] = []
for namespace_path in experiment_namespace_paths:
    for dirpath, dirnames, filenames in os.walk(namespace_path):
        if is_experiment_archive(dirpath):
            experiment_paths.append(dirpath)
            dirnames.clear()  # Prevents further recursion into subdirectories
        
print(f'Found {len(experiment_paths)} experiment archives in {len(experiment_namespace_paths)} namespaces')
pprint(experiment_paths, max_length=5)

## Experiment Loading

Load only the experiments that match our selection criteria (Experiment 09).

In [ ]:
# This list will be populated with the actual Experiment instances which will 
# be loaded from the experiment archive folders.
experiments: list[Experiment] = []

print('Loading experiments from archives...')
time_start: float = time.time()
for experiment_path in experiment_paths:
    
    experiment_identifier: str = os.path.basename(experiment_path)
    
    experiment_data_path = os.path.join(experiment_path, Experiment.DATA_FILE_NAME)
    if not os.path.exists(experiment_data_path):
        continue
    
    experiment_meta_path = os.path.join(experiment_path, Experiment.METADATA_FILE_NAME)
    if not os.path.exists(experiment_meta_path):
        continue
    
    with open(experiment_meta_path) as file:
        content = file.read()
        experiment_metadata: dict = json.loads(content)
        
    if 'parameters' not in experiment_metadata:
        continue
    
    experiment_parameters: dict = {
        param: info['value']
        for param, info in experiment_metadata['parameters'].items()
        if 'value' in info
    }
    
    # Apply filter to determine whether experiment should be included
    condition: bool = select_experiment(
        experiment_name=experiment_metadata['name'],
        experiment_metadata=experiment_metadata,
        experiment_parameters=experiment_parameters
    )
    
    if condition:
        try:
            print(f'   > included experiment "{experiment_identifier}"')
            experiment: Experiment = Experiment.load(experiment_path)
            experiments.append(experiment)
        except Exception as e:
            print(f'   Warning: Failed to load experiment "{experiment_identifier}" - Exception: {e}')
            
time_end: float = time.time()
duration: float = time_end - time_start
print(f'\nLoaded {len(experiments)} experiments in {duration:.2f} seconds')

In [ ]:
# Inspect example experiment data structure
if experiments:
    example_experiment: Experiment = experiments[0]
    print("Example experiment data structure:")
    print(f"Name: {example_experiment.metadata['name']}")
    print(f"\nParameters (sample):")
    for key in ['SEED', 'DATASET_NAME', 'NUM_REPETITIONS', 'NUM_QUERY_ACTIVES', 'EMBEDDING_SIZE', 'FP_SIZE']:
        if key in example_experiment.parameters:
            print(f"  {key}: {example_experiment.parameters[key]}")
    print(f"\nData keys:")
    pprint(list(example_experiment.data.keys()), max_length=15)
    
    # Check for metrics data
    print(f"\nMetrics keys (if available):")
    metrics_keys = [k for k in example_experiment.data.keys() if 'metrics' in k.lower()]
    pprint(metrics_keys, max_length=20)

## Experiment Sorting

Group experiments by encoding method and embedding size.

In [ ]:
# This will be a dictionary mapping the unique key of the experiment to a list of
# experiments which share that key.
key_experiment_map: dict[tuple, list[Experiment]] = defaultdict(list)

for experiment in experiments:
    key: tuple = sort_experiment(experiment)
    key_experiment_map[key].append(experiment)
    
# Sort by encoding name and then by embedding size
key_experiment_map = dict(sorted(
    key_experiment_map.items(), 
    key=lambda item: (item[0][0], item[0][1])  # (encoding, size)
))
    
print(f"\nGrouped experiments into {len(key_experiment_map)} unique configurations")
print("\nConfigurations found:")
for (encoding, size), exps in key_experiment_map.items():
    print(f"  {encoding:8s} | size={size:5d} | n_experiments={len(exps)}")

## Extract Bioactivity Prediction Metrics

Extract per-target AUC, BEDROC, and enrichment factor values from each experiment.

In [ ]:
# Collect per-target metric values for each configuration
metrics_data = {}

for (encoding, emb_size), _experiments in key_experiment_map.items():
    
    # Collect ALL per-target metric values across all experiments
    auc_values = []
    bedroc_values = []
    ef1_values = []
    ef5_values = []
    
    for exp in _experiments:
        # Load the per_target_results.csv from the experiment archive
        per_target_path = os.path.join(exp.path, 'per_target_results.csv')
        
        if os.path.exists(per_target_path):
            df = pd.read_csv(per_target_path)
            
            # Extract per-target metric values
            if 'auc_mean' in df.columns:
                auc_values.extend(df['auc_mean'].astype(float).tolist())
            
            if 'bedroc_mean' in df.columns:
                bedroc_values.extend(df['bedroc_mean'].astype(float).tolist())
            
            if 'ef1_mean' in df.columns:
                ef1_values.extend(df['ef1_mean'].astype(float).tolist())
            
            if 'ef5_mean' in df.columns:
                ef5_values.extend(df['ef5_mean'].astype(float).tolist())
        else:
            print(f"  Warning: No per_target_results.csv found in {exp.path}")
    
    # Store all per-target metrics
    metrics_data[(encoding, emb_size)] = {
        'auc': auc_values,
        'bedroc': bedroc_values,
        'ef1': ef1_values,
        'ef5': ef5_values,
    }
    
    print(f"{encoding:8s} | size={emb_size:5d} | n_targets={len(auc_values)}")

print(f"\nExtracted metrics for {len(metrics_data)} configurations")

## Box Plot: AUC vs Embedding Size

Create a box plot showing the distribution of AUC values across all targets for each embedding size and encoding method. **Higher values indicate better ranking performance.**

In [ ]:
%matplotlib inline

# ============================================================================
# CONFIGURATION
# ============================================================================

# Embedding sizes to visualize (should match sweep in _slurm_ex_09.py)
EMBEDDING_SIZES = [32, 128, 512, 2048]

# Figure size (width, height) in inches
FIGSIZE = (8, 5)

# Font size
FONT_SIZE = 11

# Color scheme
COLORS = {
    'fp': '#7C92FF',       # Blue for fingerprints
    'hdc': '#69F7AE',      # Green for HDC
}

# Labels for legend
LABELS = {
    'fp': 'Morgan FP (ECFP4)',
    'hdc': 'HDC (Depth 2)',
}

# ============================================================================
# PLOT SETUP
# ============================================================================

plt.style.use('default')
plt.rcParams['font.family'] = 'Roboto Condensed'
plt.rcParams['font.size'] = FONT_SIZE

fig, ax = plt.subplots(figsize=FIGSIZE)

# ============================================================================
# DATA PREPARATION FOR BOX PLOTS
# ============================================================================

encodings = ['fp', 'hdc']
box_width = 0.35  # Width of each box
positions_offset = [-box_width/2, box_width/2]  # Offsets for side-by-side boxes

# For each embedding size, create box plots for each encoding
for size_idx, emb_size in enumerate(EMBEDDING_SIZES):
    
    for enc_idx, encoding in enumerate(encodings):
        key = (encoding, emb_size)
        
        if key in metrics_data and metrics_data[key]['auc']:
            auc_values = metrics_data[key]['auc']
            
            # Calculate position for this box
            position = size_idx + positions_offset[enc_idx]
            
            # Create box plot
            bp = ax.boxplot(
                [auc_values],
                positions=[position],
                widths=box_width * 0.85,
                patch_artist=True,
                showfliers=True,
                boxprops=dict(facecolor=COLORS[encoding], alpha=0.95, edgecolor='black', linewidth=1.5),
                medianprops=dict(color='black', linewidth=1.5),
                whiskerprops=dict(color='black', linewidth=1.5),
                capprops=dict(color='black', linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor='black', markersize=4, alpha=0.5),
            )

# ============================================================================
# AXIS CONFIGURATION
# ============================================================================

# Set x-axis labels to embedding sizes
ax.set_xticks(range(len(EMBEDDING_SIZES)))
ax.set_xticklabels(EMBEDDING_SIZES)
ax.set_xlabel('Embedding Size', fontsize=FONT_SIZE + 1)

# Set y-axis label
ax.set_ylabel('AUC (per target) $\\uparrow$', fontsize=FONT_SIZE + 1)

# Set title
ax.set_title('Virtual Screening AUC by Embedding Size',
             fontsize=FONT_SIZE + 2, pad=15)

# Add random baseline
ax.axhline(0.5, color='red', linestyle='--', alpha=0.7, linewidth=1.5, label='Random')

# Set y-axis limits
ax.set_ylim(0.3, 1.0)

# Add grid for readability
ax.grid(True, alpha=0.3, axis='y')

# Create custom legend
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
legend_elements = [
    Patch(facecolor=COLORS['fp'], alpha=0.95, edgecolor='black', label=LABELS['fp']),
    Patch(facecolor=COLORS['hdc'], alpha=0.95, edgecolor='black', label=LABELS['hdc']),
    Line2D([0], [0], color='red', linestyle='--', linewidth=1.5, label='Random'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=FONT_SIZE)

plt.tight_layout()

# Save figure
fig_dir = os.path.join(PATH, 'figures')
os.makedirs(fig_dir, exist_ok=True)
fig_path = os.path.join(fig_dir, 'ex09_auc_boxplot.pdf')
plt.savefig(fig_path, bbox_inches='tight', dpi=300)
print(f"Saved figure to {fig_path}")

plt.show()

print("\nInterpretation:")
print("  - Higher AUC = better ranking of actives vs decoys")
print("  - AUC = 0.5 indicates random performance")
print("  - AUC = 1.0 indicates perfect ranking")

In [ ]:
%matplotlib inline

# ============================================================================
# BOX PLOT: BEDROC vs Embedding Size
# ============================================================================

plt.style.use('default')
plt.rcParams['font.family'] = 'Roboto Condensed'
plt.rcParams['font.size'] = FONT_SIZE

fig, ax = plt.subplots(figsize=FIGSIZE)

# For each embedding size, create box plots for each encoding
for size_idx, emb_size in enumerate(EMBEDDING_SIZES):
    
    for enc_idx, encoding in enumerate(encodings):
        key = (encoding, emb_size)
        
        if key in metrics_data and metrics_data[key]['bedroc']:
            bedroc_values = metrics_data[key]['bedroc']
            
            # Calculate position for this box
            position = size_idx + positions_offset[enc_idx]
            
            # Create box plot
            bp = ax.boxplot(
                [bedroc_values],
                positions=[position],
                widths=box_width * 0.85,
                patch_artist=True,
                showfliers=True,
                boxprops=dict(facecolor=COLORS[encoding], alpha=0.95, edgecolor='black', linewidth=1.5),
                medianprops=dict(color='black', linewidth=1.5),
                whiskerprops=dict(color='black', linewidth=1.5),
                capprops=dict(color='black', linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor='black', markersize=4, alpha=0.5),
            )

# Axis configuration
ax.set_xticks(range(len(EMBEDDING_SIZES)))
ax.set_xticklabels(EMBEDDING_SIZES)
ax.set_xlabel('Embedding Size', fontsize=FONT_SIZE + 1)
ax.set_ylabel('BEDROC(α=20) (per target) $\\uparrow$', fontsize=FONT_SIZE + 1)
ax.set_title('Virtual Screening BEDROC by Embedding Size\n(Early Recognition Metric)',
             fontsize=FONT_SIZE + 2, pad=15)

# Add random baseline
ax.axhline(0.0, color='red', linestyle='--', alpha=0.7, linewidth=1.5, label='Random')

ax.grid(True, alpha=0.3, axis='y')

# Legend
legend_elements = [
    Patch(facecolor=COLORS['fp'], alpha=0.95, edgecolor='black', label=LABELS['fp']),
    Patch(facecolor=COLORS['hdc'], alpha=0.95, edgecolor='black', label=LABELS['hdc']),
    Line2D([0], [0], color='red', linestyle='--', linewidth=1.5, label='Random'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=FONT_SIZE)

plt.tight_layout()

# Save figure
fig_path = os.path.join(fig_dir, 'ex09_bedroc_boxplot.pdf')
plt.savefig(fig_path, bbox_inches='tight', dpi=300)
print(f"Saved figure to {fig_path}")

plt.show()

print("\nInterpretation:")
print("  - Higher BEDROC = better early recognition of actives")
print("  - BEDROC emphasizes finding actives in top-ranked compounds")
print("  - Critical metric for practical virtual screening")

## LaTeX Table: Summary Statistics

Generate a LaTeX table with mean and standard deviation of all metrics for each configuration.

In [ ]:
# ============================================================================
# GENERATE LATEX TABLE
# ============================================================================

# Build table data
table_rows = []

for emb_size in EMBEDDING_SIZES:
    row = {'Embedding Size': emb_size}
    
    for encoding in ['fp', 'hdc']:
        key = (encoding, emb_size)
        
        if key in metrics_data and metrics_data[key]['auc']:
            auc_values = metrics_data[key]['auc']
            bedroc_values = metrics_data[key]['bedroc']
            ef1_values = metrics_data[key]['ef1']
            ef5_values = metrics_data[key]['ef5']
            
            row[f'{encoding.upper()} AUC'] = f'{np.mean(auc_values):.3f} $\\pm$ {np.std(auc_values):.3f}'
            row[f'{encoding.upper()} BEDROC'] = f'{np.mean(bedroc_values):.3f} $\\pm$ {np.std(bedroc_values):.3f}'
            if ef1_values:
                row[f'{encoding.upper()} EF1%'] = f'{np.mean(ef1_values):.2f} $\\pm$ {np.std(ef1_values):.2f}'
            if ef5_values:
                row[f'{encoding.upper()} EF5%'] = f'{np.mean(ef5_values):.2f} $\\pm$ {np.std(ef5_values):.2f}'
        else:
            row[f'{encoding.upper()} AUC'] = 'N/A'
            row[f'{encoding.upper()} BEDROC'] = 'N/A'
    
    table_rows.append(row)

# Create DataFrame
df_table = pd.DataFrame(table_rows)

# Display as regular table first
print("Summary Statistics (higher is better for all metrics):")
print("="*80)
display(df_table)

In [ ]:
# Generate LaTeX
latex_table = r"""
\begin{table}[htbp]
\centering
\caption{Bioactivity prediction performance by embedding size. Higher values indicate better performance.}
\label{tab:bioactivity_emb_size}
\begin{tabular}{rcccc}
\toprule
\textbf{Emb. Size} & \textbf{FP AUC} & \textbf{FP BEDROC} & \textbf{HDC AUC} & \textbf{HDC BEDROC} \\
\midrule
"""

for row in table_rows:
    fp_auc = row.get('FP AUC', 'N/A')
    fp_bedroc = row.get('FP BEDROC', 'N/A')
    hdc_auc = row.get('HDC AUC', 'N/A')
    hdc_bedroc = row.get('HDC BEDROC', 'N/A')
    latex_table += f"{row['Embedding Size']} & {fp_auc} & {fp_bedroc} & {hdc_auc} & {hdc_bedroc} \\\\\n"

latex_table += r"""\bottomrule
\end{tabular}
\end{table}
"""

print("\n" + "="*80)
print("LaTeX Table:")
print("="*80)
print(latex_table)

# Save LaTeX to file
latex_path = os.path.join(PATH, 'figures', 'ex09_bioactivity_table.tex')
with open(latex_path, 'w') as f:
    f.write(latex_table)
print(f"\nSaved LaTeX table to {latex_path}")

## Statistical Comparison: FP vs HDC

Perform statistical tests to quantify differences between representation methods at each embedding size.

In [ ]:
from scipy import stats

print("\n" + "="*80)
print("STATISTICAL COMPARISON: HDC vs FP (higher is better)")
print("="*80)

# For each embedding size, compare FP and HDC
for emb_size in EMBEDDING_SIZES:
    print(f"\n\nEmbedding Size: {emb_size}")
    print("-" * 40)
    
    # Get AUC values for each method
    fp_auc = metrics_data.get(('fp', emb_size), {}).get('auc', [])
    hdc_auc = metrics_data.get(('hdc', emb_size), {}).get('auc', [])
    
    if not (fp_auc and hdc_auc):
        print("  Warning: Insufficient data for statistical comparison")
        continue
    
    # Print means
    print(f"  FP:      {np.mean(fp_auc):.4f} +/- {np.std(fp_auc):.4f}")
    print(f"  HDC:     {np.mean(hdc_auc):.4f} +/- {np.std(hdc_auc):.4f}")
    
    # Perform paired t-test (if same number of samples) or independent t-test
    if len(fp_auc) == len(hdc_auc):
        t_stat, p_value = stats.ttest_rel(hdc_auc, fp_auc)
        test_type = "paired"
    else:
        t_stat, p_value = stats.ttest_ind(hdc_auc, fp_auc)
        test_type = "independent"
    
    # Determine which is better (higher AUC is better)
    diff = np.mean(hdc_auc) - np.mean(fp_auc)
    better = "HDC" if diff > 0 else "FP"
    
    sig = "***" if p_value < 0.001 else "**" if p_value < 0.01 else "*" if p_value < 0.05 else "ns"
    print(f"\n  {test_type.capitalize()} t-test: t={t_stat:.3f}, p={p_value:.4f} {sig}")
    print(f"  Better method: {better} (diff = {diff:+.4f})")

print("\n\nSignificance levels: *** p<0.001, ** p<0.01, * p<0.05, ns = not significant")

## Line Plot: Mean Metrics vs Embedding Size

Line plot showing mean AUC and BEDROC across embedding sizes for easier trend identification.

In [ ]:
%matplotlib inline

# ============================================================================
# LINE PLOT: MEAN AUC AND BEDROC TRENDS
# ============================================================================

plt.style.use('default')
plt.rcParams['font.family'] = 'Roboto Condensed'
plt.rcParams['font.size'] = FONT_SIZE

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Plot AUC
ax = axes[0]
for encoding in ['fp', 'hdc']:
    means = []
    stds = []
    sizes_available = []
    
    for emb_size in EMBEDDING_SIZES:
        key = (encoding, emb_size)
        if key in metrics_data and metrics_data[key]['auc']:
            auc_values = metrics_data[key]['auc']
            means.append(np.mean(auc_values))
            stds.append(np.std(auc_values))
            sizes_available.append(emb_size)
    
    if means:
        means = np.array(means)
        stds = np.array(stds)
        sizes_available = np.array(sizes_available)
        
        ax.errorbar(sizes_available, means, yerr=stds, 
                   fmt='o-', label=LABELS[encoding], 
                   color=COLORS[encoding], capsize=5, linewidth=2, markersize=8,
                   markeredgecolor='black', markeredgewidth=1.5)

ax.axhline(0.5, color='red', linestyle='--', alpha=0.7, linewidth=1.5, label='Random')
ax.set_xlabel('Embedding Size', fontsize=FONT_SIZE + 1)
ax.set_ylabel('Mean AUC $\\uparrow$', fontsize=FONT_SIZE + 1)
ax.set_title('AUC vs Embedding Size\n(Mean +/- Std across targets)',
             fontsize=FONT_SIZE + 2, pad=15)
ax.legend(fontsize=FONT_SIZE)
ax.grid(True, alpha=0.3)
ax.set_xscale('log', base=2)
ax.set_xticks(EMBEDDING_SIZES)
ax.set_xticklabels(EMBEDDING_SIZES)

# Plot BEDROC
ax = axes[1]
for encoding in ['fp', 'hdc']:
    means = []
    stds = []
    sizes_available = []
    
    for emb_size in EMBEDDING_SIZES:
        key = (encoding, emb_size)
        if key in metrics_data and metrics_data[key]['bedroc']:
            bedroc_values = metrics_data[key]['bedroc']
            means.append(np.mean(bedroc_values))
            stds.append(np.std(bedroc_values))
            sizes_available.append(emb_size)
    
    if means:
        means = np.array(means)
        stds = np.array(stds)
        sizes_available = np.array(sizes_available)
        
        ax.errorbar(sizes_available, means, yerr=stds, 
                   fmt='o-', label=LABELS[encoding], 
                   color=COLORS[encoding], capsize=5, linewidth=2, markersize=8,
                   markeredgecolor='black', markeredgewidth=1.5)

ax.axhline(0.0, color='red', linestyle='--', alpha=0.7, linewidth=1.5, label='Random')
ax.set_xlabel('Embedding Size', fontsize=FONT_SIZE + 1)
ax.set_ylabel('Mean BEDROC(α=20) $\\uparrow$', fontsize=FONT_SIZE + 1)
ax.set_title('BEDROC vs Embedding Size\n(Mean +/- Std across targets)',
             fontsize=FONT_SIZE + 2, pad=15)
ax.legend(fontsize=FONT_SIZE)
ax.grid(True, alpha=0.3)
ax.set_xscale('log', base=2)
ax.set_xticks(EMBEDDING_SIZES)
ax.set_xticklabels(EMBEDDING_SIZES)

plt.tight_layout()

# Save figure
fig_path = os.path.join(fig_dir, 'ex09_metrics_lineplot.pdf')
plt.savefig(fig_path, bbox_inches='tight', dpi=300)
print(f"Saved figure to {fig_path}")

plt.show()

print("\nInterpretation:")
print("  - Upward trend = performance improves with embedding size")
print("  - Plateau = diminishing returns at higher dimensions")
print("  - Error bars show variance across different targets")

## Summary

Key findings from this analysis.

In [ ]:
print("="*80)
print("EXPERIMENT 09 SUMMARY: Bioactivity Prediction vs Embedding Size")
print("="*80)

# Get dataset name from first experiment
dataset_name = "Unknown"
if experiments:
    dataset_name = experiments[0].parameters.get('DATASET_NAME', 'Unknown')

print(f"\nDataset: {dataset_name}")
print(f"Embedding sizes tested: {EMBEDDING_SIZES}")
print("Encoding methods: Morgan FP (ECFP4), HDC (Depth 2)")

print("\n" + "-"*40)
print("KEY FINDINGS:")
print("-"*40)

# Find best configuration for each method
for encoding in ['fp', 'hdc']:
    best_size = None
    best_auc = 0
    
    for emb_size in EMBEDDING_SIZES:
        key = (encoding, emb_size)
        if key in metrics_data and metrics_data[key]['auc']:
            mean_auc = np.mean(metrics_data[key]['auc'])
            if mean_auc > best_auc:
                best_auc = mean_auc
                best_size = emb_size
    
    if best_size:
        best_bedroc = np.mean(metrics_data[(encoding, best_size)]['bedroc'])
        print(f"\n{LABELS[encoding]}:")
        print(f"  Best embedding size: {best_size}")
        print(f"  Best mean AUC: {best_auc:.4f}")
        print(f"  Corresponding BEDROC: {best_bedroc:.4f}")

print("\n" + "="*80)